# Session 16: Project 3 — Customer Segmentation with AI-Generated Personas

Segment online retail customers into distinct groups using **RFM analysis**
and **K-means clustering**, then use **Google Gemini** to automatically
generate marketing personas for each segment.

| Part | What You Do |
|------|-------------|
| Part 1 | Load & explore the online retail dataset |
| Part 2 | Complete TODOs 1-3 in `segmentation_engine.py`: data cleaning and RFM features |
| Part 3 | Complete TODOs 4-6: feature scaling and K-means clustering |
| Part 4 | Complete TODOs 7-8: visualization and cluster analysis |
| Part 5 | Complete TODOs 9-10: AI personas and the engine class |
| Part 6 | Build the Streamlit dashboard (TODOs 11-15 in `segmentation_app.py`) |
| Part 7 | Reflection & next steps |

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load API key from .env file (see .env.example)
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print('python-dotenv not installed. Install with: pip install python-dotenv')
    print('Or set GOOGLE_API_KEY as an environment variable manually.')

API_KEY = os.environ.get('GOOGLE_API_KEY')
if API_KEY:
    print('API key loaded from .env')
else:
    print('No API key found. Persona generation will be skipped.')
    print('To set up: copy .env.example to .env and add your key.')

---
## Part 1: Load & Explore the Dataset

The dataset contains ~8,000 online retail transactions. Each row is
one line item from an invoice — a customer buying a certain quantity
of a product at a given price.

In [ ]:
# Load the raw data
df = pd.read_csv('data/online_retail_sample.csv')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# Data types and structure
df.info()

In [ ]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\n{df["CustomerID"].isnull().sum()} rows have no CustomerID '
      f'({df["CustomerID"].isnull().mean():.1%} of data)')

In [ ]:
# Quick data quality checks
print(f'Negative Quantity rows (returns): {(df["Quantity"] < 0).sum()}')
print(f'Zero/negative UnitPrice rows:     {(df["UnitPrice"] <= 0).sum()}')
print(f'Unique customers (non-null):       {df["CustomerID"].nunique()}')
print(f'Unique invoices:                   {df["InvoiceNo"].nunique()}')
print(f'Date range:                        {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')

In [ ]:
# Transactions over time
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
monthly = df.set_index('InvoiceDate').resample('M').size()

plt.figure(figsize=(10, 4))
monthly.plot(kind='bar', color='steelblue', alpha=0.8)
plt.title('Transactions per Month')
plt.xlabel('Month')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## Part 2: Data Cleaning & RFM Features

Open `segmentation_engine.py` and complete **TODOs 1-3**.

### TODO 1: `load_and_clean(filepath)`
Clean the raw transactions: drop missing CustomerIDs, remove returns
and zero-price items, add a TotalPrice column.

### TODOs 2-3: `build_rfm(df)`
Create one row per customer with three features.

### Why RFM? Real-world context

RFM analysis has been a standard customer segmentation technique in
marketing and CRM since the 1990s. It's used by companies ranging
from e-commerce retailers to subscription services to banks.

The reason it works in practice:

- **Recency** captures engagement — customers who bought recently are
  more likely to respond to campaigns. Direct mail marketers discovered
  this decades ago: response rates drop sharply with time since last purchase.

- **Frequency** captures loyalty — repeat buyers are cheaper to retain
  than new customers are to acquire. The rule of thumb is that acquiring
  a new customer costs 5-7x more than retaining an existing one.

- **Monetary** captures value — not all customers are equally profitable.
  The Pareto principle applies: roughly 20% of customers drive 80% of revenue.

RFM is used in real businesses for segment-specific campaigns
(e.g., win-back emails for high-Recency customers, loyalty rewards for
high-Frequency buyers, VIP programs for high-Monetary spenders),
customer lifetime value estimation, and churn prediction.

Many companies extend RFM with additional features — average order value,
product category diversity, return rate, time between purchases — but the
core three are the starting point that most analytics teams use.

After completing your TODOs, test below.

In [ ]:
# Reload the module after editing (run this after saving your changes)
import importlib
import segmentation_engine
importlib.reload(segmentation_engine)

from segmentation_engine import load_and_clean

clean_df = load_and_clean('data/online_retail_sample.csv')
print(f'Cleaned: {len(clean_df)} transactions (removed {len(df) - len(clean_df)} invalid rows)')
clean_df.head()

In [ ]:
from segmentation_engine import build_rfm

rfm = build_rfm(clean_df)
print(f'RFM table: {len(rfm)} customers')
rfm.head(10)

In [ ]:
# Visualize RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes,
                          ['Recency', 'Frequency', 'Monetary'],
                          ['steelblue', 'darkorange', 'seagreen']):
    ax.hist(rfm[col], bins=30, color=color, alpha=0.7, edgecolor='black')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

### RFM Interpretation

**What do these distributions tell us?**

- **Recency**: Right-skewed — most customers haven't purchased recently.
  A small group bought very recently (low recency = engaged).

- **Frequency**: Heavily right-skewed — most customers made only 1-3
  purchases. A few power users stand out with many orders.

- **Monetary**: Also right-skewed — a small number of high-value customers
  drive a disproportionate share of revenue (the Pareto effect in action).

These skewed distributions are typical in real retail data and are
exactly why segmentation is valuable — treating all customers the same
wastes marketing budget.

---
## Part 3: Feature Scaling & K-Means Clustering

Open `segmentation_engine.py` and complete **TODOs 4-6**.

- **TODO 4**: `scale_features()` — StandardScaler on the RFM columns
- **TODO 5**: `find_optimal_k()` — try K=2 through 8, record inertia + silhouette
- **TODO 6**: `run_kmeans()` — fit KMeans with a chosen K

In [ ]:
importlib.reload(segmentation_engine)
from segmentation_engine import scale_features

scaled_data, scaler = scale_features(rfm)
print(f'Scaled shape: {scaled_data.shape}')
print(f'Means (should be ~0): {scaled_data.mean(axis=0).round(4)}')
print(f'Stds  (should be ~1): {scaled_data.std(axis=0).round(4)}')

In [ ]:
from segmentation_engine import find_optimal_k

k_results = find_optimal_k(scaled_data)
k_results

### Choosing K

Look for:
- The **elbow** in the inertia curve (where it bends)
- The **highest silhouette score** (closer to 1 = better-defined clusters)

Pick a K that balances both. In real projects, business context matters too —
marketing teams typically want 3-5 actionable segments, not 8.

In [ ]:
from segmentation_engine import run_kmeans

# Pick your K based on the results above
n_clusters = 4  # Change this based on your analysis

kmeans_model = run_kmeans(scaled_data, n_clusters)
rfm['Cluster'] = kmeans_model.labels_

print(f'Customers per cluster:')
print(rfm['Cluster'].value_counts().sort_index())

---
## Part 4: Visualization & Cluster Analysis

Open `segmentation_engine.py` and complete **TODOs 7-8**.

- **TODO 7**: `plot_elbow_and_silhouette()` — side-by-side K selection plots
- **TODO 8**: `plot_clusters_2d()` — PCA projection colored by cluster

In [ ]:
importlib.reload(segmentation_engine)
from segmentation_engine import plot_elbow_and_silhouette

plot_elbow_and_silhouette(k_results)

In [ ]:
from segmentation_engine import plot_clusters_2d

plot_clusters_2d(scaled_data, kmeans_model.labels_, rfm)

In [ ]:
# Cluster profiles: mean RFM values per cluster
cluster_profiles = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].agg(['mean', 'count'])
print('Cluster profiles:')
cluster_profiles

In [ ]:
# Cluster profile bar chart
cluster_means = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes,
                          ['Recency', 'Frequency', 'Monetary'],
                          ['steelblue', 'darkorange', 'seagreen']):
    ax.bar(cluster_means.index, cluster_means[col],
           color=color, alpha=0.8, edgecolor='black')
    ax.set_xlabel('Cluster')
    ax.set_ylabel(f'Mean {col}')
    ax.set_title(f'{col} by Cluster')

plt.tight_layout()
plt.show()

---
## Part 5: AI-Powered Personas with Gemini

Open `segmentation_engine.py` and complete **TODOs 9-10**.

- **TODO 9**: `generate_persona()` — prompt Gemini with cluster stats
- **TODO 10**: `SegmentationEngine.run_segmentation()` — wire the full pipeline

Your API key should be loaded from your `.env` file (see setup cell above).
If you don't have a key yet, copy `.env.example` to `.env` and add your key.

In [ ]:
importlib.reload(segmentation_engine)
from segmentation_engine import generate_persona

# Test persona generation for one cluster
if API_KEY:
    from google import genai
    client = genai.Client(api_key=API_KEY)

    cluster_stats = rfm[rfm['Cluster'] == 0][['Recency', 'Frequency', 'Monetary']].mean()
    persona = generate_persona(cluster_stats, 0, client)
    print('Cluster 0 Persona:')
    print(persona)
else:
    print('Skipping — no API key configured.')
    print('Copy .env.example to .env and add your GOOGLE_API_KEY.')

In [ ]:
# Test the full SegmentationEngine class
from segmentation_engine import SegmentationEngine

engine = SegmentationEngine('data/online_retail_sample.csv')
engine.run_segmentation(n_clusters=4)

In [ ]:
# Generate personas for all clusters (requires API key in .env)
if engine.client:
    engine.generate_personas()
else:
    print('Persona generation skipped — no API key.')

In [ ]:
# View cluster summary
engine.get_cluster_summary()

---
## Part 6: Build the Streamlit Dashboard

Open `segmentation_app.py` and complete **TODOs 11-15**:

- **TODO 11**: Initialize session state for the engine
- **TODO 12**: Handle the Run button (create engine, run segmentation)
- **TODO 13**: Data Overview tab (metrics + RFM histograms)
- **TODO 14**: Cluster Results tab (summary table + PCA plot + profiles)
- **TODO 15**: Persona Generation tab (Gemini integration)

The app reads your API key from `.env` automatically — no need to paste it in the UI.

Run with:
```bash
streamlit run segmentation_app.py
```

Or test the reference solution:
```bash
streamlit run solution/segmentation_app.py
```

---
## Part 7: Reflection & Next Steps

Answer these questions in the cell below:

1. How many clusters did you choose and why?
2. Which cluster is the most valuable to the business? Why?
3. What marketing action would you recommend for each segment?
4. What additional features (beyond RFM) might improve the segmentation?
5. How would you explain your segmentation results to a non-technical stakeholder?

In [ ]:
# Write your reflections here
#
# 1. Number of clusters:
#
# 2. Most valuable cluster:
#
# 3. Marketing recommendations:
#
# 4. Additional features:
#
# 5. Stakeholder explanation:
#